In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
#
# Passing `message.content` back unchanged (instead of rebuilding the blocks by
# hand) is the pattern the docs recommend: it keeps every block type intact,
# including the thinking blocks that current models emit between tool calls.
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=None,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
):
    params = {
        "model": model,
        "max_tokens": 2000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    if tool_choice:
        params["tool_choice"] = tool_choice

    return client.messages.stream(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Tool definition
#
# `eager_input_streaming: True` enables fine-grained tool streaming for this
# tool: the API streams the tool's input as Claude generates it, without
# server-side buffering, so large arguments (like the review below) start
# arriving immediately instead of all at once when the block closes.
# This per-tool field is GA and replaces the legacy
# `fine-grained-tool-streaming-2025-05-14` beta header. The trade-off: the
# fragments are not validated server-side, so a truncated response can leave
# the tool input incomplete — run_tools (next cell) reports any failure back
# to Claude with is_error instead of crashing.

save_article_schema = {
    "name": "save_article",
    "description": "Saves a scholarly journal article",
    "eager_input_streaming": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "Abstract of the article. One short sentence max",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Word count",
                    },
                    "review": {
                        "type": "string",
                        "description": "Eight sentence review of the paper",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
}


def save_article(**kwargs):
    return "Article saved!"

In [4]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
# Run conversation
#
# The stream helper emits SDK events: "text" for text deltas, "input_json" for
# tool-input fragments (these arrive incrementally thanks to
# eager_input_streaming), plus the raw content_block_start/stop events.
def run_conversation(messages, tools=None, tool_choice=None):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            tool_choice=tool_choice,
        ) as stream:
            for event in stream:
                if event.type == "text":
                    print(event.text, end="")

                if event.type == "content_block_start":
                    if event.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{event.content_block.name}"')

                if event.type == "input_json" and event.partial_json:
                    print(event.partial_json, end="")

                if event.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        # A forced tool_choice makes Claude call the tool on every turn, so
        # send the results back once and stop instead of looping forever.
        if tool_choice:
            break

    return messages

In [6]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(messages, tools=[save_article_schema])

I

'll create a fake computer science article and save it for you.


>>> Tool Call: "save_article"
{"abstract": "A novel qu

antum-inspired neural architecture reduces training time for large language models by 47% while maintaining accu

racy.", "meta": {"word_count": 6842, "review": "This paper presents an intriguing fusion of quantum computing principles and traditional neural network architectures, proposing what

 the authors call 'Quantum-Entangled Gradient Descent' (QEGD). The the

...

led Gradient Descent" (QEGD) — a fictional technique combining quantum computing principles with neural network training
- **Review high

lights:** The review notes the creative theoretical framing, promising experimental results, but also flags limitations like the l

ack of real quantum hardware testing and insufficient discussion of computational overhead from classical simulation of quantum states.

Let me know if you'd like

 me to generate another article on a different topic or with a different focus (e.g., cybersecurity, distributed systems, or AI ethics)!



[{'role': 'user',
  'content': 'Create and save a fake computer science article'},
 {'role': 'assistant',
  'content': [ParsedTextBlock(citations=None, text="I'll create a fake computer science article and save it for you.", type='text', parsed_output=None),
   ToolUseBlock(id='toolu_01XppbfqMXx2HnTwopV2tCR7', caller=DirectCaller(type='direct'), input={'abstract': 'A novel quantum-inspired neural architecture reduces training time for large language models by 47% while maintaining accuracy.', 'meta': {'word_count': 6842, 'review': "This paper presents an intriguing fusion of quantum computing principles and traditional neural network architectures, proposing what the authors call 'Quantum-Entangled Gradient Descent' (QEGD). The theoretical foundation draws on superposition states to explore multiple gradient paths simultaneously, which is a creative reimagining of standard backpropagation. The experimental results, showing a 47% reduction in training time across three benchmark dataset